In [1]:
import os
import pandas as pd

from pathlib import Path
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Project paths
project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

load_dotenv(project_root / ".env", override=True)

API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YouTube API key not found in .env")

youtube = build("youtube", "v3", developerKey=API_KEY)

# ---------------------------------------
# CONFIGURATION
# ---------------------------------------

VIDEO_URLS = [
    "https://www.youtube.com/watch?v=VIDEO_ID_1",
    "https://www.youtube.com/watch?v=VIDEO_ID_2",
    "https://www.youtube.com/watch?v=VIDEO_ID_3",
]

MAX_COMMENTS_PER_VIDEO = 100

# Change the batch name for future collection rounds
BATCH_NAME = "batch01"

# ---------------------------------------
# EXTRACT VIDEO ID
# ---------------------------------------

def extract_video_id(url):
    parsed = urlparse(url)
    host = parsed.netloc.lower().removeprefix("www.")

    if host == "youtu.be":
        return parsed.path.strip("/").split("/")[0]

    if host in {"youtube.com", "m.youtube.com"}:
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in {"shorts", "live"}:
            return parts[1]

    raise ValueError(f"Invalid YouTube URL: {url}")


# ---------------------------------------
# COLLECT TOP-LEVEL COMMENTS
# ---------------------------------------

def collect_comments(video_url, max_comments=100):
    video_id = extract_video_id(video_url)

    # Use one consistent video link format
    video_link = f"https://www.youtube.com/watch?v={video_id}"

    records = []
    next_page_token = None

    while len(records) < max_comments:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, max_comments - len(records)),
                pageToken=next_page_token,
                textFormat="plainText",
                order="time"
            ).execute()

        except HttpError as error:
            print(
                f"Could not collect from {video_id}: "
                f"HTTP {error.resp.status}"
            )
            break

        for item in response.get("items", []):
            comment = item["snippet"]["topLevelComment"]
            snippet = comment["snippet"]

            records.append({
                "comment_id": comment["id"],
                "text": snippet["textDisplay"],
                "video_id": video_id,
                "video_link": video_link
            })

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

    print(f"Collected {len(records)} comments from {video_link}")
    return records


# ---------------------------------------
# COLLECT FROM ALL VIDEOS
# ---------------------------------------

all_comments = []

for url in VIDEO_URLS:
    all_comments.extend(
        collect_comments(url, MAX_COMMENTS_PER_VIDEO)
    )

df = pd.DataFrame(
    all_comments,
    columns=["comment_id", "text", "video_id", "video_link"]
)

if df.empty:
    raise ValueError("No comments collected. Check your video URLs.")

# Clean whitespace while preserving emojis and punctuation
df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["text"] != ""].copy()

# Remove duplicate comment IDs, keeping identical text
# from different video contexts when relevant
df = df.drop_duplicates(subset=["comment_id"])

# Save collection CSV
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / f"{BATCH_NAME}_youtube_comments.csv"

df.to_csv(raw_path, index=False, encoding="utf-8-sig")

print(f"\nSaved CSV: {raw_path}")
print(f"Unique comments collected: {len(df)}")

display(df.head())

Could not collect from VIDEO_ID_1: HTTP 404
Collected 0 comments from https://www.youtube.com/watch?v=VIDEO_ID_1
Could not collect from VIDEO_ID_2: HTTP 404
Collected 0 comments from https://www.youtube.com/watch?v=VIDEO_ID_2
Could not collect from VIDEO_ID_3: HTTP 404
Collected 0 comments from https://www.youtube.com/watch?v=VIDEO_ID_3


ValueError: No comments collected. Check your video URLs.

In [2]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

# Number of comments to annotate
# Use None to include every collected comment
SAMPLE_SIZE = None

annotation_source = df.copy()

if SAMPLE_SIZE is not None:
    annotation_source = annotation_source.sample(
        n=min(SAMPLE_SIZE, len(annotation_source)),
        random_state=42
    )

annotation_source = annotation_source.reset_index(drop=True)

# Create annotation-ready dataframe
annotation_df = pd.DataFrame({
    "ID": [
        f"{BATCH_NAME}-{i:04d}"
        for i in range(1, len(annotation_source) + 1)
    ],
    "text": annotation_source["text"],
    "video_link": annotation_source["video_link"],
    "label": "",
    "confidence": "",
    "notes": ""
})

output_dir = project_root / "data" / "annotated"
output_dir.mkdir(parents=True, exist_ok=True)

# Save annotation-ready CSV as well
annotation_df.to_csv(
    output_dir / f"{BATCH_NAME}_annotation_template.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save two sheets in one Excel workbook
output_path = output_dir / f"{BATCH_NAME}_annotation.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for person in ["Person_1", "Person_2"]:
        annotation_df.to_excel(
            writer,
            sheet_name=person,
            index=False
        )

wb = load_workbook(output_path)

for ws in wb.worksheets:

    # Label dropdown (column D)
    label_dropdown = DataValidation(
        type="list",
        formula1='"Sincere,Sarcasm/Irony,Uncertain"',
        allow_blank=True
    )

    ws.add_data_validation(label_dropdown)
    label_dropdown.add("D2:D10000")

    # Confidence dropdown (column E)
    confidence_dropdown = DataValidation(
        type="list",
        formula1='"1 - Low,2 - Medium,3 - High"',
        allow_blank=True
    )

    ws.add_data_validation(confidence_dropdown)
    confidence_dropdown.add("E2:E10000")

    # Header formatting
    for cell in ws[1]:
        cell.fill = PatternFill(
            start_color="17365D",
            end_color="17365D",
            fill_type="solid"
        )
        cell.font = Font(color="FFFFFF", bold=True)

    # Column widths
    widths = {
        "A": 18,  # ID
        "B": 85,  # Text
        "C": 55,  # Video link
        "D": 22,  # Label
        "E": 22,  # Confidence
        "F": 55   # Notes
    }

    for column, width in widths.items():
        ws.column_dimensions[column].width = width

    # Make links clickable and keep comment text literal
    for row in range(2, ws.max_row + 1):
        text_cell = ws[f"B{row}"]
        text_cell.data_type = "s"

        link_cell = ws[f"C{row}"]

        if link_cell.value:
            link_cell.hyperlink = str(link_cell.value)
            link_cell.style = "Hyperlink"

        for cell in ws[row]:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

        ws.row_dimensions[row].height = 45

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb.save(output_path)

print(f"Excel workbook created: {output_path}")
print(f"Comments per annotator: {len(annotation_df)}")
print("Both sheets include video links and dropdowns.")

Excel workbook created: c:\Users\84905\Documents\GitHub\FYP-Vietnamese-Sarcasm\data\annotated\batch01_annotation.xlsx
Comments per annotator: 0
Both sheets include video links and dropdowns.
